In [ ]:
import json
import argparse
import pandas as pd
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, precision_score, recall_score, roc_curve
import scipy
import numpy as np

def thr_to_accuracy(thr, Y_test, predictions):
   return -f1_score(Y_test, np.array(predictions>thr, dtype=int), average='weighted')

def calc_score(y, y_hat, fn, **kwargs):
    if len(y_hat) == 0:
        return 0
    return fn(y, y_hat, **kwargs)

metrics = [json.loads(line) for line in open('../metrics/runs-small-weighted-graph.jsonl')]
metrics = pd.DataFrame(metrics)

metrics['threshold'] = metrics.apply(lambda x: scipy.optimize.fmin(thr_to_accuracy, args=(x['ground_truth'], x['probas']), x0=0.4), axis=1)
metrics['predictions'] = metrics.apply(lambda x: (np.array(x['probas']) > x['threshold']).astype(int).tolist(), axis=1)

metrics['roc_auc'] = metrics.apply(lambda x:  calc_score(x['ground_truth'], x['probas'], roc_auc_score, average='weighted'), axis=1)
metrics['f1'] = metrics.apply(lambda x: calc_score(x['ground_truth'], x['predictions'], f1_score, average='weighted'), axis=1)
metrics['precision'] = metrics.apply(lambda x: calc_score(x['ground_truth'], x['predictions'], precision_score, average='weighted'), axis=1)
metrics['recall'] = metrics.apply(lambda x: calc_score(x['ground_truth'], x['predictions'], recall_score, average='weighted'), axis=1)
metrics['accuracy'] = metrics.apply(lambda x: calc_score(x['ground_truth'], x['predictions'], accuracy_score), axis=1)

In [ ]:
g = metrics[metrics['ratio'].isin([0.50, 0.70])]
g = g.groupby(['ratio', 'model']).agg({'f1': ['mean', 'std'], 'precision': ['mean', 'std'], 'recall': ['mean', 'std']
                                    #    , 'accuracy': ['mean', 'std'], 'roc_auc': ['mean', 'std']
                                       })
g = g[[('f1', 'mean'), ('f1', 'std'), ('precision', 'mean'), ('precision', 'std'), ('recall', 'mean'), ('recall', 'std'), 
    #    ('accuracy', 'mean'), ('accuracy', 'std'), ('roc_auc', 'mean'), ('roc_auc', 'std')
       ]].groupby(['ratio'], group_keys=False)
print(g.apply(lambda x: x.sort_values(by=('f1', 'mean'), ascending=False)).round(4).to_latex())

In [ ]:
g = metrics.groupby(['ratio', 'model'])[['f1', 'precision', 'recall']].mean()
g = g[['f1', 'precision', 'recall']].groupby(['ratio'], group_keys=False)
print(g.apply(lambda x: x.sort_values(by='f1', ascending=False)))